# 02.01 — Fundamentos de Apache Spark con PySpark

## Objetivo

Este notebook introduce los conceptos fundamentales necesarios para entender
cómo Apache Spark procesa datos.

A diferencia de trabajar únicamente con las operaciones de una API, el objetivo
es observar también qué ocurre internamente durante la ejecución.

Se estudiarán los siguientes conceptos:

- `SparkSession`
- DataFrames distribuidos
- Schemas
- Transformations y Actions
- Lazy Evaluation
- Partitions
- `repartition()` y `coalesce()`
- Planes de ejecución con `explain()`
- Relación entre Jobs, Stages y Tasks

La meta es comenzar a pensar en Spark como un motor de procesamiento distribuido,
y no simplemente como una alternativa a pandas.

In [1]:
# Importaciones
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

## 1. Creación de la SparkSession

Toda aplicación de PySpark necesita un punto de entrada para interactuar con Spark. Actualmente, este punto de entrada es SparkSession.

Desde esta sesión podremos crear DataFrames, leer y escribir archivos, ejecutar consultas SQL y acceder a diferentes configuraciones del motor de Spark.

In [2]:
# Crear la SparkSesion
spark = (
    SparkSession.builder
    .appName("02_01_spark_fundamentals")
    .getOrCreate()
)

spark

# 2. Información del entorno Spark

Antes de comenzar a procesar datos, es útil conocer cómo está ejecutándose nuestra aplicación. En esta sección consultaremos algunos datos básicos del entorno, como la versión de Spark, el nombre de la aplicación, el modo de ejecución y el nivel de paralelismo disponible.

In [3]:
print("=" * 60)
print("INFORMACIÓN DEL ENTORNO SPARK")
print("=" * 60)

print(f"Spark version: {spark.version}")
print(f"Application name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Application ID: {spark.sparkContext.applicationId}")

INFORMACIÓN DEL ENTORNO SPARK
Spark version: 4.1.3
Application name: 02_01_spark_fundamentals
Master: local[*]
Default parallelism: 12
Application ID: local-1790537898612


**Resultado obtenido**

El valor local[*] indica que estamos ejecutando Spark en modo local y que puede utilizar todos los núcleos lógicos que tenga disponibles el contenedor.

El defaultParallelism = 12 nos indica que Spark dispone inicialmente de un nivel de paralelismo de 12 tareas para determinadas operaciones. Más adelante veremos cómo este valor se relaciona con las particiones y los Tasks.

## 3. Configuración básica de Spark

Spark cuenta con numerosas propiedades de configuración que controlan cómo se ejecutan las aplicaciones. Por ahora revisaremos únicamente algunas de las más importantes para entender nuestro entorno.

In [4]:
print("Configuraciones importantes:\n")

configs = [
    "spark.master",
    "spark.app.name",
    "spark.sql.shuffle.partitions"
]

for config in configs:
    try:
        print(f"{config}: {spark.conf.get(config)}")
    except Exception:
        print(f"{config}: no disponible")

Configuraciones importantes:

spark.master: local[*]
spark.app.name: 02_01_spark_fundamentals
spark.sql.shuffle.partitions: 200


spark.sql.shuffle.partitions establece el número predeterminado de particiones utilizadas después de ciertas operaciones que redistribuyen datos entre particiones, como agregaciones y joins.

Por ahora mantendremos el valor predeterminado. Más adelante experimentaremos con él para comprender su impacto.

## 4. Creación de un DataFrame

Un DataFrame de Spark representa una colección de datos organizada en columnas.

Visualmente puede parecer similar a un DataFrame de pandas, pero existe una
diferencia fundamental:

**un DataFrame de Spark está diseñado para poder distribuir sus datos y su
procesamiento entre diferentes particiones y máquinas.**

En esta primera prueba utilizaremos un dataset pequeño para concentrarnos
únicamente en la estructura y comportamiento de Spark.

In [5]:
data = [
    (1, "CDMX", 25, 15000.0),
    (2, "Jalisco", 34, 22000.0),
    (3, "Nuevo León", 41, 31000.0),
    (4, "CDMX", 29, 18000.0),
    (5, "Puebla", 52, 27000.0),
    (6, "Jalisco", 38, 24500.0),
    (7, "CDMX", 45, 35000.0),
    (8, "Nuevo León", 31, 21000.0),
]

columns = [
    "id",
    "estado",
    "edad",
    "ingreso"
]

## 5. Construcción del DataFrame

Ahora transformaremos los datos de Python en un DataFrame administrado por Spark.

In [6]:
df = spark.createDataFrame(
    data,
    schema=columns
)

df.show()

+---+----------+----+-------+
| id|    estado|edad|ingreso|
+---+----------+----+-------+
|  1|      CDMX|  25|15000.0|
|  2|   Jalisco|  34|22000.0|
|  3|Nuevo León|  41|31000.0|
|  4|      CDMX|  29|18000.0|
|  5|    Puebla|  52|27000.0|
|  6|   Jalisco|  38|24500.0|
|  7|      CDMX|  45|35000.0|
|  8|Nuevo León|  31|21000.0|
+---+----------+----+-------+



show() nos permite visualizar una muestra del contenido del DataFrame. Más adelante veremos que esta operación también tiene una implicación importante dentro del modelo de ejecución de Spark.

## 6. Inspección del schema

Cada DataFrame posee un schema, que describe la estructura de los datos: nombres de columnas, tipos y posibilidad de contener valores nulos.

Vamos a inspeccionar el schema que Spark generó automáticamente a partir de nuestros datos.

In [7]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- estado: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- ingreso: double (nullable = true)



Como no proporcionamos los tipos explícitamente, Spark tuvo que inferirlos a partir de los valores disponibles.

## 7. Definición explícita del schema

En proyectos reales suele ser conveniente definir explícitamente la estructura esperada de los datos. Esto nos permite controlar los tipos de cada columna y evitar depender completamente de la inferencia automática de Spark.

In [8]:
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("estado", StringType(), False),
    StructField("edad", IntegerType(), True),
    StructField("ingreso", DoubleType(), True)
])

df_schema = spark.createDataFrame(
    data,
    schema=schema
)

df_schema.printSchema()

root
 |-- id: integer (nullable = false)
 |-- estado: string (nullable = false)
 |-- edad: integer (nullable = true)
 |-- ingreso: double (nullable = true)



En cada StructField especificamos:

$$\text{nombre de columna}\quad \rightarrow \quad \text{tipo de dato} \quad \rightarrow \quad \text{permite o no valores null}$$

Por ejemplo:

StructField("id", IntegerType(), False)

indica que id será un entero y no deberá aceptar valores nulos.

## 8. Inspección básica del DataFrame

Antes de comenzar a realizar transformaciones, revisaremos algunas propiedades básicas del DataFrame que acabamos de crear.

In [9]:
print("Columnas:")
print(df.columns)

print("\nNúmero de columnas:")
print(len(df.columns))

print("\nSchema:")
print(df.schema)

Columnas:
['id', 'estado', 'edad', 'ingreso']

Número de columnas:
4

Schema:
StructType([StructField('id', LongType(), True), StructField('estado', StringType(), True), StructField('edad', LongType(), True), StructField('ingreso', DoubleType(), True)])


Con esto podemos consultar la estructura del DataFrame sin comenzar todavía nuestro experimento sobre el modelo de ejecución de Spark.

## 9. Transformations vs Actions

Una de las ideas más importantes para entender Spark es que no todas las instrucciones que escribimos provocan inmediatamente un procesamiento de los datos.

Spark distingue principalmente entre dos tipos de operaciones:

- **Transformations:** describen cómo queremos transformar los datos.

- **Actions:** solicitan a Spark producir o devolver un resultado.

En esta sección construiremos varias operaciones paso a paso para observar esta diferencia.

### Transformaciones

In [10]:
df_transformado = (
    df
    .filter(F.col("edad") >= 30)
    .select(
        "id",
        "estado",
        "edad",
        "ingreso"
    )
    .withColumn(
        "ingreso_anual",
        F.col("ingreso") * 12
    )
)

df_transformado

DataFrame[id: bigint, estado: string, edad: bigint, ingreso: double, ingreso_anual: double]

En este punto hemos definido tres transformaciones:

```text
df
 │
 ├── filter()
 │
 ▼
select()
 │
 ▼
withColumn()
 │
 ▼
df_transformado
```



Sin embargo, todavía no hemos solicitado a Spark que nos entregue los registros resultantes.

El objeto `df_transformado` representa el conjunto de operaciones que queremos realizar.

### Algunas transformaciones comunes

Entre las operaciones que normalmente funcionan como transformaciones encontramos:

```python
df.select(...)
df.filter(...)
df.withColumn(...)
df.drop(...)
df.distinct(...)
df.groupBy(...)
df.join(...)
df.orderBy(...)
```

Una característica importante es que estas operaciones producen un nuevo DataFrame.

Por ejemplo:

In [11]:
df_filtrado = df.filter(F.col("edad") >= 30)

`df` continúa existiendo sin modificaciones.

Podemos comprobarlo:

In [12]:
print("Columnas del DataFrame original:")
print(df.columns)

print("\nColumnas del DataFrame transformado:")
print(df_transformado.columns)

Columnas del DataFrame original:
['id', 'estado', 'edad', 'ingreso']

Columnas del DataFrame transformado:
['id', 'estado', 'edad', 'ingreso', 'ingreso_anual']


### Actions

- ### `Show()`

`show()` es una **Action**.

En este momento Spark necesita obtener los datos para poder mostrarlos, por lo que debe ejecutar las transformaciones que definimos anteriormente.

Conceptualmente:

```text
filter()
    ↓
select()
    ↓
withColumn()
    ↓
    ↓
 show()
    ↓
EJECUCIÓN
```

In [13]:
df_transformado.show()

+---+----------+----+-------+-------------+
| id|    estado|edad|ingreso|ingreso_anual|
+---+----------+----+-------+-------------+
|  2|   Jalisco|  34|22000.0|     264000.0|
|  3|Nuevo León|  41|31000.0|     372000.0|
|  5|    Puebla|  52|27000.0|     324000.0|
|  6|   Jalisco|  38|24500.0|     294000.0|
|  7|      CDMX|  45|35000.0|     420000.0|
|  8|Nuevo León|  31|21000.0|     252000.0|
+---+----------+----+-------+-------------+



### Diferentes tipos de Actions

Spark dispone de varias acciones dependiendo del resultado que necesitemos obtener.

Probemos algunas de ellas.

- ### `count()`

Devuelve el número de registros.

In [14]:
numero_registros = df_transformado.count()

print(f"Número de registros: {numero_registros}")

Número de registros: 6


- ### `first()`

Obtiene el primer registro.

In [15]:
primer_registro = df_transformado.first()

print(primer_registro)

Row(id=2, estado='Jalisco', edad=34, ingreso=22000.0, ingreso_anual=264000.0)


- ### `take()`

Obtiene una cantidad específica de registros.

In [16]:
primeros_tres = df_transformado.take(3)

primeros_tres

[Row(id=2, estado='Jalisco', edad=34, ingreso=22000.0, ingreso_anual=264000.0),
 Row(id=3, estado='Nuevo León', edad=41, ingreso=31000.0, ingreso_anual=372000.0),
 Row(id=5, estado='Puebla', edad=52, ingreso=27000.0, ingreso_anual=324000.0)]

Observa que el resultado ya no es un DataFrame de Spark, sino una lista de objetos `Row` de Python.

Podemos comprobarlo:

In [17]:
print(type(primeros_tres))
print(type(primeros_tres[0]))

<class 'list'>
<class 'pyspark.sql.types.Row'>


- ### `collect()`

`collect()` recupera **todos los registros del DataFrame y los lleva al proceso Driver**.


In [18]:
datos_locales = df_transformado.collect()

datos_locales

[Row(id=2, estado='Jalisco', edad=34, ingreso=22000.0, ingreso_anual=264000.0),
 Row(id=3, estado='Nuevo León', edad=41, ingreso=31000.0, ingreso_anual=372000.0),
 Row(id=5, estado='Puebla', edad=52, ingreso=27000.0, ingreso_anual=324000.0),
 Row(id=6, estado='Jalisco', edad=38, ingreso=24500.0, ingreso_anual=294000.0),
 Row(id=7, estado='CDMX', edad=45, ingreso=35000.0, ingreso_anual=420000.0),
 Row(id=8, estado='Nuevo León', edad=31, ingreso=21000.0, ingreso_anual=252000.0)]

En nuestro ejemplo esto no representa ningún problema porque solamente tenemos unos cuantos registros.

Sin embargo, en Big Data debemos utilizar `collect()` con cuidado.

Conceptualmente:

```text
Executor / Partition
        │
Executor / Partition
        │
Executor / Partition
        │
        ▼
      DRIVER
```

Si tenemos millones de registros, mover todos los datos al Driver puede consumir toda su memoria.

Por ello:

```python
df.collect()
```

puede ser completamente válido para:

```text
10 registros
100 registros
1,000 registros
```

pero potencialmente peligroso para un dataset de gran tamaño.

Esta diferencia entre **procesamiento distribuido** y **llevar los datos al Driver** será muy importante durante todo el proyecto.

## 10. Lazy Evaluation

Spark utiliza un modelo conocido como Lazy Evaluation o evaluación perezosa.

Cuando encadenamos transformaciones, Spark normalmente no procesa inmediatamente los datos. En su lugar, registra las operaciones y construye un plan de ejecución.

El procesamiento comienza cuando aparece una Action que necesita obtener un resultado.

Analizaremos este comportamiento construyendo un pequeño pipeline y observando cuándo se dispara realmente la ejecución.

In [19]:
pipeline = (
    df
    .filter(F.col("ingreso") > 20000)
    .withColumn(
        "ingreso_anual",
        F.col("ingreso") * 12
    )
    .withColumn(
        "grupo_edad",
        F.when(F.col("edad") < 35, "Menor de 35")
         .when(F.col("edad") < 45, "35-44")
         .otherwise("45 o más")
    )
    .select(
        "id",
        "estado",
        "edad",
        "grupo_edad",
        "ingreso",
        "ingreso_anual"
    )
)

Hasta aquí solamente hemos definido:

```text
DataFrame original
       │
       ▼
     filter
       │
       ▼
  withColumn
       │
       ▼
  withColumn
       │
       ▼
     select
```

No necesitamos escribir:

```python
pipeline.show()
```

todavía.

Primero podemos inspeccionar el objeto:

In [20]:
pipeline

DataFrame[id: bigint, estado: string, edad: bigint, grupo_edad: string, ingreso: double, ingreso_anual: double]

Spark conoce las transformaciones que debe realizar, pero no necesitamos haber materializado todavía el resultado completo.

### 10.1 Inspeccionar el plan antes de ejecutar una Action

Una ventaja importante de este modelo es que Spark puede analizar todas las transformaciones antes de realizar el cálculo.

Podemos inspeccionar el plan con:

In [21]:
pipeline.explain()

== Physical Plan ==
*(1) Project [id#0L, estado#1, edad#2L, CASE WHEN (edad#2L < 35) THEN Menor de 35 WHEN (edad#2L < 45) THEN 35-44 ELSE 45 o más END AS grupo_edad#48, ingreso#3, (ingreso#3 * 12.0) AS ingreso_anual#47]
+- *(1) Filter (isnotnull(ingreso#3) AND (ingreso#3 > 20000.0))
   +- *(1) Scan ExistingRDD[id#0L,estado#1,edad#2L,ingreso#3]




Obtendremos un resultado parecido a:

```text
== Physical Plan ==
*(1) Project [...]
+- *(1) Filter [...]
   +- *(1) Scan ExistingRDD [...]
```

No necesitamos comprender todavía todos los elementos del plan.

Por ahora identifica únicamente algunas palabras:

```text
Filter
Project
Scan
```

Estas representan operaciones que Spark planea realizar.

Podemos interpretarlas aproximadamente como:

```text
Scan
  ↓
leer datos

Filter
  ↓
filtrar registros

Project
  ↓
seleccionar/calcular columnas
```


### 10.2 Ejecutar finalmente el pipeline

Ahora sí utilizamos una Action.

In [22]:
pipeline.show(truncate=False)

+---+----------+----+-----------+-------+-------------+
|id |estado    |edad|grupo_edad |ingreso|ingreso_anual|
+---+----------+----+-----------+-------+-------------+
|2  |Jalisco   |34  |Menor de 35|22000.0|264000.0     |
|3  |Nuevo León|41  |35-44      |31000.0|372000.0     |
|5  |Puebla    |52  |45 o más   |27000.0|324000.0     |
|6  |Jalisco   |38  |35-44      |24500.0|294000.0     |
|7  |CDMX      |45  |45 o más   |35000.0|420000.0     |
|8  |Nuevo León|31  |Menor de 35|21000.0|252000.0     |
+---+----------+----+-----------+-------+-------------+



En este momento Spark toma el plan que construyó y ejecuta las operaciones necesarias para producir el resultado.

Podemos resumir el proceso como:

```text
Código PySpark
      │
      ▼
Transformations
      │
      ▼
Plan de ejecución
      │
      ▼
   Action
      │
      ▼
Spark ejecuta
      │
      ▼
Resultado
```

### 10.3 Un detalle importante: una Action puede generar una nueva ejecución

Vamos a realizar dos Actions distintas:

In [23]:
pipeline.show()

numero = pipeline.count()

print(f"Número de registros: {numero}")

+---+----------+----+-----------+-------+-------------+
| id|    estado|edad| grupo_edad|ingreso|ingreso_anual|
+---+----------+----+-----------+-------+-------------+
|  2|   Jalisco|  34|Menor de 35|22000.0|     264000.0|
|  3|Nuevo León|  41|      35-44|31000.0|     372000.0|
|  5|    Puebla|  52|   45 o más|27000.0|     324000.0|
|  6|   Jalisco|  38|      35-44|24500.0|     294000.0|
|  7|      CDMX|  45|   45 o más|35000.0|     420000.0|
|  8|Nuevo León|  31|Menor de 35|21000.0|     252000.0|
+---+----------+----+-----------+-------+-------------+

Número de registros: 6


Aunque ambas utilizan el mismo DataFrame, Spark puede necesitar ejecutar nuevamente parte del pipeline para cada Action.

Esto ocurre porque hasta ahora **no hemos indicado que el resultado deba conservarse en memoria**.

Más adelante estudiaremos:

```python
cache()
persist()
```

que permiten reutilizar resultados cuando un mismo DataFrame será utilizado varias veces.

Por ahora basta con recordar:

```text
Transformation
    ↓
no implica necesariamente ejecución inmediata

Action
    ↓
puede disparar un Job de Spark
```

### 10.4 Resumen: Transformations, Actions y Lazy Evaluation

Podemos resumir lo aprendido hasta ahora de la siguiente manera:

| Concepto        | Función                                                |
| --------------- | ------------------------------------------------------ |
| Transformation  | Define una operación sobre los datos                   |
| Action          | Solicita un resultado                                  |
| Lazy Evaluation | Retrasa la ejecución hasta que sea necesaria           |
| Execution Plan  | Describe cómo Spark planea realizar las operaciones    |
| Driver          | Coordina la ejecución y recibe determinados resultados |

Algunos ejemplos:

| Operación      | Tipo           |
| -------------- | -------------- |
| `select()`     | Transformation |
| `filter()`     | Transformation |
| `withColumn()` | Transformation |
| `join()`       | Transformation |
| `groupBy()`    | Transformation |
| `show()`       | Action         |
| `count()`      | Action         |
| `first()`      | Action         |
| `take()`       | Action         |
| `collect()`    | Action         |

La idea clave es:

> En Spark, escribir una transformación describe **qué queremos hacer**. Una Action provoca que Spark determine y ejecute **cómo hacerlo**.

## 11. Particiones en Spark

Las **particiones** son las unidades en las que Spark divide un conjunto de datos para poder procesarlo en paralelo.

En lugar de tratar un DataFrame como un único bloque, Spark puede distribuir sus registros entre varias particiones. Cada partición puede ser procesada por una tarea distinta.

Conceptualmente:

```text
DataFrame
   │
   ├── Partition 0 ──► Task
   ├── Partition 1 ──► Task
   ├── Partition 2 ──► Task
   └── Partition 3 ──► Task
```

En un clúster real, estas tareas podrían incluso ejecutarse en diferentes máquinas. En nuestro entorno actual estamos trabajando en modo:

```text
local[*]
```

por lo que todo se ejecuta dentro de una sola máquina, utilizando los recursos disponibles.

### Crear un DataFrame para estudiar las particiones

Para visualizar mejor este comportamiento construiremos un DataFrame más grande utilizando `spark.range()`.

In [24]:
df_particiones = spark.range(0, 120)

df_particiones.show(10)

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+
only showing top 10 rows


`spark.range()` genera un DataFrame distribuido directamente desde Spark.

Podemos inspeccionar su schema:

In [25]:
df_particiones.printSchema()

root
 |-- id: long (nullable = false)



### 11.1 Número de particiones

Podemos consultar cuántas particiones tiene actualmente el DataFrame utilizando:

In [26]:
num_particiones = df_particiones.rdd.getNumPartitions()

print(f"Número de particiones: {num_particiones}")

Número de particiones: 12


En nuestro entorno es esperable obtener:

```text
12
```

Esto coincide con el valor que observamos anteriormente:

```text
Default parallelism: 12
```

En este caso, Spark creó el DataFrame utilizando el nivel de paralelismo disponible por defecto.

> El número de particiones no siempre será igual a `defaultParallelism`. Dependerá de cómo se hayan creado o leído los datos y de las operaciones aplicadas posteriormente.

### 11.2 Identificar la partición de cada registro

Spark proporciona la función:

In [27]:
F.spark_partition_id()

Column<'spark_partition_id()'>

que nos permite identificar la partición en la que se encuentra cada registro.

Vamos a agregar temporalmente esta información al DataFrame.

In [28]:
df_con_particion = (
    df_particiones
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
)

df_con_particion.show(30)

+---+------------+
| id|partition_id|
+---+------------+
|  0|           0|
|  1|           0|
|  2|           0|
|  3|           0|
|  4|           0|
|  5|           0|
|  6|           0|
|  7|           0|
|  8|           0|
|  9|           0|
| 10|           1|
| 11|           1|
| 12|           1|
| 13|           1|
| 14|           1|
| 15|           1|
| 16|           1|
| 17|           1|
| 18|           1|
| 19|           1|
| 20|           2|
| 21|           2|
| 22|           2|
| 23|           2|
| 24|           2|
| 25|           2|
| 26|           2|
| 27|           2|
| 28|           2|
| 29|           2|
+---+------------+
only showing top 30 rows


Podemos observar que varios registros consecutivos pertenecen a una misma partición.

### 11.3 Cantidad de registros por partición

Para visualizar mejor la distribución de los datos, contaremos cuántos registros existen dentro de cada partición.

In [29]:
distribucion_particiones = (
    df_con_particion
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)

distribucion_particiones.show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|   10|
|           1|   10|
|           2|   10|
|           3|   10|
|           4|   10|
|           5|   10|
|           6|   10|
|           7|   10|
|           8|   10|
|           9|   10|
|          10|   10|
|          11|   10|
+------------+-----+



Con 120 registros y 12 particiones es probable que observemos una distribución cercana a:

```text
+------------+-----+
|partition_id|count|
+------------+-----+
|           0|   10|
|           1|   10|
|           2|   10|
|           3|   10|
|           4|   10|
|           5|   10|
|           6|   10|
|           7|   10|
|           8|   10|
|           9|   10|
|          10|   10|
|          11|   10|
+------------+-----+
```

Esto nos permite visualizar la relación:

```text
120 registros
      │
      ▼
12 particiones
      │
      ▼
≈ 10 registros por partición
```

El objetivo de Spark es poder procesar estas particiones de manera independiente siempre que la operación lo permita.

### 11.4 Particiones y paralelismo

Una partición representa, de forma simplificada, una unidad de trabajo que puede ser procesada por una Task.

Por ejemplo:

```text
Partition 0 ──► Task 0
Partition 1 ──► Task 1
Partition 2 ──► Task 2
...
Partition 11 ─► Task 11
```

Como nuestro entorno tiene:

```text
defaultParallelism = 12
```

Spark puede aprovechar múltiples recursos de procesamiento para ejecutar varias Tasks de forma concurrente.

Sin embargo, tener más particiones **no significa automáticamente mayor rendimiento**.

Por ejemplo:

```text
1,000,000 registros
        │
        ├── 1 partición
        │
        └── 1 Task
```

podría desaprovechar recursos disponibles.

Pero:

```text
1,000,000 registros
        │
        ├── 100,000 particiones
        │
        └── 100,000 Tasks
```

también sería ineficiente debido al costo de administrar una cantidad excesiva de tareas.

Por ello, el particionamiento es una decisión importante en aplicaciones de Big Data.


## 12. Modificar el número de particiones

Spark permite modificar explícitamente el número de particiones de un DataFrame.

Dos métodos importantes son:

- `repartition()`
- `coalesce()`

Aunque ambos pueden cambiar la cantidad de particiones, su funcionamiento interno es diferente.

### 12.1 `repartition()`

Vamos a reducir nuestro DataFrame de 12 a 6 particiones.

In [30]:
df_repartition = df_particiones.repartition(6)

print(
    "Número de particiones después de repartition:",
    df_repartition.rdd.getNumPartitions()
)

Número de particiones después de repartition: 6


Podemos volver a observar dónde quedaron los registros.

In [31]:
df_repartition_dist = (
    df_repartition
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
)

(
    df_repartition_dist
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
    .show()
)

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|   22|
|           1|   18|
|           2|   19|
|           3|   21|
|           4|   19|
|           5|   21|
+------------+-----+



Ahora Spark redistribuyó los registros entre seis particiones.

Conceptualmente:

```text
ANTES

P0   P1   P2   P3   ...   P11

          ↓

      repartition(6)

          ↓

P0   P1   P2   P3   P4   P5
```

`repartition()` redistribuye los datos y normalmente requiere un **shuffle**.

### 12.2 ¿Qué es un shuffle?

Un **shuffle** ocurre cuando Spark necesita mover datos entre particiones.

De forma simplificada:

```text
Partition 0 ─────┐
Partition 1 ──┐  │
Partition 2 ──┼──┼──► nuevas particiones
Partition 3 ──┘  │
Partition 4 ─────┘
```

Mover datos tiene un costo porque puede implicar:

- serialización;
- uso de memoria;
- escritura temporal en disco;
- transferencia de datos;
- coordinación entre tareas.

En nuestro entorno local no existe transferencia entre máquinas, pero el concepto será exactamente el mismo cuando posteriormente trabajemos con un clúster.

Los **shuffles** serán uno de los principales aspectos que aprenderemos a identificar y reducir al optimizar aplicaciones Spark.


### 12.3 `coalesce()`

Ahora probaremos otra forma de reducir el número de particiones.

In [32]:
df_coalesce = df_particiones.coalesce(3)

print(
    "Número de particiones después de coalesce:",
    df_coalesce.rdd.getNumPartitions()
)

Número de particiones después de coalesce: 3


Veamos la distribución:

In [33]:
df_coalesce_dist = (
    df_coalesce
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
)

(
    df_coalesce_dist
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
    .show()
)

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|   40|
|           1|   40|
|           2|   40|
+------------+-----+



## 13. `repartition()` vs `coalesce()`


Aunque ambos métodos modifican el número de particiones, existe una diferencia importante.

### `repartition()`

`repartition()` redistribuye los datos entre las nuevas particiones.

```text
Particiones originales
      │
      ▼
    SHUFFLE
      │
      ▼
Nuevas particiones
```

Esto suele producir una distribución más equilibrada, pero tiene un costo adicional.

Puede utilizarse tanto para **aumentar como para disminuir** el número de particiones.

Por ejemplo:

In [36]:
df.repartition(20)

DataFrame[id: bigint, estado: string, edad: bigint, ingreso: double]

### `coalesce()`

`coalesce()` está pensado principalmente para **reducir** el número de particiones.

En lugar de redistribuir completamente todos los datos, intenta combinar particiones existentes.

```text
P0 ──┐
P1 ──┘──► P0

P2 ──┐
P3 ──┘──► P1
```

Esto normalmente evita un shuffle completo y, por tanto, puede resultar más económico.

Sin embargo, existe una desventaja: las nuevas particiones pueden quedar menos equilibradas.

---

### Comparación

| Característica | `repartition()` | `coalesce()` |
|---|---|---|
| Aumentar particiones | Sí | No es su uso habitual |
| Reducir particiones | Sí | Sí |
| Redistribución completa | Sí | Generalmente no |
| Shuffle | Generalmente sí | Generalmente no |
| Balance de datos | Mejor | Puede ser desigual |
| Costo | Mayor | Menor |

Una regla práctica inicial es:

```text
Necesito redistribuir los datos
           ↓
      repartition()

Solo quiero reducir particiones
           ↓
        coalesce()
```

Más adelante veremos que la decisión real depende del tamaño de los datos y de lo que queramos hacer posteriormente.

### 13.1 Comparación experimental

Podemos comparar los tres DataFrames que hemos construido.

In [38]:
print(
    "Particiones originales:",
    df_particiones.rdd.getNumPartitions()
)

print(
    "Después de repartition(6):",
    df_repartition.rdd.getNumPartitions()
)

print(
    "Después de coalesce(3):",
    df_coalesce.rdd.getNumPartitions()
)

Particiones originales: 12
Después de repartition(6): 6
Después de coalesce(3): 3


Con esto podemos comprobar que un mismo conjunto de datos puede representarse utilizando distintas cantidades de particiones.

## 14. ¿Por qué importan las particiones en Big Data?

Las particiones afectan directamente la forma en la que Spark distribuye el trabajo.

Muy pocas particiones pueden provocar:

```text
poco paralelismo
      ↓
recursos desaprovechados
      ↓
ejecución más lenta
```

Mientras que demasiadas particiones pueden provocar:

```text
demasiadas Tasks
      ↓
mayor overhead
      ↓
más coordinación
      ↓
menor eficiencia
```

Por ello, cuando trabajemos con datasets grandes no nos interesará únicamente conocer:

```text
¿Cuántos registros tengo?
```

También será importante preguntarnos:

```text
¿Cómo están distribuidos?

¿Cuántas particiones existen?

¿Son aproximadamente equilibradas?

¿Estoy provocando shuffles innecesarios?
```

Estas preguntas serán fundamentales cuando lleguemos a optimización de Spark.

## 15. Resumen de particiones

Hasta ahora podemos relacionar los conceptos de la siguiente manera:

```text
DataFrame
    │
    ▼
Partitions
    │
    ▼
Tasks
    │
    ▼
procesamiento paralelo
```

Y hemos introducido dos operaciones importantes:

```text
repartition()
    │
    └── redistribuye datos
          └── normalmente produce shuffle


coalesce()
    │
    └── combina particiones
          └── normalmente evita shuffle completo
```

La idea fundamental es:

> Spark obtiene su capacidad de procesamiento paralelo dividiendo los datos en particiones que pueden ser procesadas por diferentes Tasks.

## 16. Planes de ejecución con `explain()`

Hasta ahora hemos escrito transformaciones y observado sus resultados. Sin embargo, Spark también nos permite inspeccionar **cómo planea ejecutar esas operaciones**.

Para ello utilizaremos:

In [39]:
df.explain("formatted")

== Physical Plan ==
* Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [4]: [id#0L, estado#1, edad#2L, ingreso#3]
Arguments: [id#0L, estado#1, edad#2L, ingreso#3], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)




El objetivo no es memorizar cada elemento del plan, sino comenzar a reconocer operaciones importantes y relacionarlas con nuestro código.


### Inspeccionar un pipeline sencillo

Utilizaremos el pipeline que construimos anteriormente:

In [40]:
pipeline.explain("formatted")

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [4]: [id#0L, estado#1, edad#2L, ingreso#3]
Arguments: [id#0L, estado#1, edad#2L, ingreso#3], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [4]: [id#0L, estado#1, edad#2L, ingreso#3]
Condition : (isnotnull(ingreso#3) AND (ingreso#3 > 20000.0))

(3) Project [codegen id : 1]
Output [6]: [id#0L, estado#1, edad#2L, CASE WHEN (edad#2L < 35) THEN Menor de 35 WHEN (edad#2L < 45) THEN 35-44 ELSE 45 o más END AS grupo_edad#48, ingreso#3, (ingreso#3 * 12.0) AS ingreso_anual#47]
Input [4]: [id#0L, estado#1, edad#2L, ingreso#3]




Dependiendo de la versión y configuración de Spark, el resultado puede variar ligeramente, pero deberían aparecer operaciones similares a:

```text
Scan ExistingRDD
      ↓
Filter
      ↓
Project
```

Podemos interpretarlas de forma simplificada como:

| Operación | Significado aproximado |
|---|---|
| `Scan` | Leer los datos de origen |
| `Filter` | Filtrar registros |
| `Project` | Seleccionar o calcular columnas |

Nuestro código:

```python
pipeline = (
    df
    .filter(F.col("ingreso") > 20000)
    .withColumn("ingreso_anual", F.col("ingreso") * 12)
    .withColumn(...)
    .select(...)
)
```

termina convertido por Spark en un plan físico de ejecución.

### 16.1 Spark no ejecuta literalmente cada línea de nuestro código

Un aspecto importante es que el plan físico no tiene por qué contener una operación independiente por cada instrucción escrita.

Por ejemplo:

```python
df.withColumn(...)
  .withColumn(...)
  .select(...)
```

puede terminar representado por una cantidad menor de operaciones `Project`.

Esto ocurre porque Spark analiza el conjunto completo de transformaciones antes de ejecutar los datos y puede simplificar u optimizar el plan.

Conceptualmente:

```text
Código escrito por nosotros
         ↓
   Logical Plan
         ↓
Optimized Logical Plan
         ↓
    Physical Plan
         ↓
      Ejecución
```

Esta optimización es posible, en gran parte, gracias al modelo de **Lazy Evaluation** que estudiamos anteriormente.

### 16.2 Observar un `repartition()` en el plan

Anteriormente ejecutamos:

```python
df_repartition = df_particiones.repartition(6)
```

Ahora inspeccionaremos su plan.

In [41]:
df_repartition.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (6)
+- == Final Plan ==
   ResultQueryStage (4)
   +- ShuffleQueryStage (3), Statistics(sizeInBytes=1920.0 B, rowCount=120)
      +- Exchange (2)
         +- * Range (1)
+- == Initial Plan ==
   Exchange (5)
   +- Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#97L]
Arguments: Range (0, 120, step=1, splits=Some(12))

(2) Exchange
Input [1]: [id#97L]
Arguments: RoundRobinPartitioning(6), REPARTITION_BY_NUM, [plan_id=293]

(3) ShuffleQueryStage
Output [1]: [id#97L]
Arguments: 0

(4) ResultQueryStage
Output [1]: [id#97L]
Arguments: 1

(5) Exchange
Input [1]: [id#97L]
Arguments: RoundRobinPartitioning(6), REPARTITION_BY_NUM, [plan_id=288]

(6) AdaptiveSparkPlan
Output [1]: [id#97L]
Arguments: isFinalPlan=true




Entre las operaciones debería aparecer algo similar a:

```text
Exchange
```

y posiblemente una descripción como:

```text
RoundRobinPartitioning(6)
```

La parte importante es:

```text
Exchange
```

En un plan de Spark, `Exchange` normalmente indica que los datos deben ser **redistribuidos entre particiones**.

Es decir:

```text
Partition 0 ─────┐
Partition 1 ──┐  │
Partition 2 ──┼──┼──► Exchange ──► nuevas particiones
Partition 3 ──┘  │
Partition 4 ─────┘
```

Por eso `repartition()` suele producir un **shuffle**.


### 16.3 Comparar con `coalesce()`

Ahora observemos:

In [42]:
df_coalesce.explain("formatted")

== Physical Plan ==
Coalesce (2)
+- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#97L]
Arguments: Range (0, 120, step=1, splits=Some(12))

(2) Coalesce
Input [1]: [id#97L]
Arguments: 3




Recordemos que construimos:

```python
df_coalesce = df_particiones.coalesce(3)
```

Aquí deberíamos encontrar una operación relacionada con:

```text
Coalesce
```

pero normalmente no un `Exchange` equivalente al generado por `repartition()`.

Esta diferencia refleja lo que vimos anteriormente:

```text
repartition()
     ↓
redistribución de datos
     ↓
Exchange / Shuffle


coalesce()
     ↓
combina particiones existentes
     ↓
sin redistribución completa
```

Esto hace que `coalesce()` pueda resultar más barato cuando únicamente queremos reducir el número de particiones.


## 17. Un shuffle producido por una agregación

Los shuffles no aparecen únicamente cuando utilizamos explícitamente `repartition()`.

Muchas operaciones habituales necesitan redistribuir datos.

Un ejemplo clásico es:

```python
groupBy()
```

Vamos a construir una agregación utilizando nuestro primer DataFrame.

In [43]:
df_por_estado = (
    df
    .groupBy("estado")
    .agg(
        F.count("*").alias("num_registros"),
        F.avg("ingreso").alias("ingreso_promedio")
    )
)

Todavía no ejecutaremos:

```python
df_por_estado.show()
```

Primero inspeccionaremos el plan.

In [44]:
df_por_estado.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [4]: [id#0L, estado#1, edad#2L, ingreso#3]
Arguments: [id#0L, estado#1, edad#2L, ingreso#3], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [2]: [estado#1, ingreso#3]
Input [4]: [id#0L, estado#1, edad#2L, ingreso#3]

(3) HashAggregate
Input [2]: [estado#1, ingreso#3]
Keys [1]: [estado#1]
Functions [2]: [partial_count(1), partial_avg(ingreso#3)]
Aggregate Attributes [3]: [count#165L, sum#166, count#167L]
Results [4]: [estado#1, count#168L, sum#169, count#170L]

(4) Exchange
Input [4]: [estado#1, count#168L, sum#169, count#170L]
Arguments: hashpartitioning(estado#1, 200), ENSURE_REQUIREMENTS, [plan_id=505]

(5) HashAggregate
Input [4]: [estado#1, count#168L, sum#169, count#170L]
Keys [1]: [estado#1]
Fun

Busca nuevamente una operación:

```text
Exchange
```

Es posible que también aparezca algo parecido a:

```text
hashpartitioning(estado, 200)
```

### 17.1 ¿Por qué `groupBy()` necesita mover datos?

Supongamos que inicialmente los registros se encuentran así:

```text
Partition 0
CDMX
Jalisco
Puebla

Partition 1
CDMX
Jalisco
Nuevo León

Partition 2
CDMX
Puebla
Nuevo León
```

Si queremos calcular:

```python
df.groupBy("estado").count()
```

Spark necesita reunir los registros correspondientes a la misma clave.

Conceptualmente:

```text
ANTES DEL SHUFFLE

P0 → CDMX, Jalisco, Puebla
P1 → CDMX, Jalisco, Nuevo León
P2 → CDMX, Puebla, Nuevo León

              ↓
           SHUFFLE
              ↓

DESPUÉS

P? → CDMX, CDMX, CDMX
P? → Jalisco, Jalisco
P? → Puebla, Puebla
P? → Nuevo León, Nuevo León
```

Solo después de reunir los datos por clave puede completar correctamente la agregación.

### 17.2 Relación con `spark.sql.shuffle.partitions`

Anteriormente observamos:

```text
spark.sql.shuffle.partitions = 200
```

Este parámetro establece el número predeterminado de particiones utilizadas en determinadas operaciones de shuffle de Spark SQL y DataFrames.

Por eso podemos encontrar en el plan algo parecido a:

```text
hashpartitioning(estado, 200)
```

aunque nuestro DataFrame solo tenga ocho registros.

Esto parece exagerado para un ejemplo pequeño, y efectivamente lo sería si Spark tuviera que ejecutar literalmente 200 particiones finales sin ninguna optimización adicional.

Sin embargo, las versiones modernas de Spark pueden aplicar optimizaciones dinámicas durante la ejecución mediante **Adaptive Query Execution (AQE)**.

Por ahora no modificaremos esta configuración. La estudiaremos con más detalle cuando lleguemos a optimización.


## 18. Adaptive Query Execution

Al inspeccionar algunos planes es posible encontrar:

```text
AdaptiveSparkPlan
```

Por ejemplo:

```text
AdaptiveSparkPlan isFinalPlan=false
```

Esto significa que Spark puede modificar ciertas decisiones del plan físico mientras ejecuta la consulta utilizando información real sobre los datos.

De forma simplificada:

```text
Plan inicial
     ↓
Spark comienza a ejecutar
     ↓
obtiene estadísticas reales
     ↓
puede ajustar el plan
     ↓
Plan adaptado
```

Una de las cosas que AQE puede hacer es combinar particiones de shuffle demasiado pequeñas.

No profundizaremos todavía en este mecanismo, pero es importante reconocer el término cuando aparezca en nuestros planes.

## 19. Narrow vs Wide Transformations

Ahora podemos introducir una distinción muy importante entre transformaciones.

### Narrow Transformations

Una transformación es **narrow** cuando cada partición de salida depende de una cantidad limitada de particiones de entrada y no requiere redistribuir globalmente los datos.

Ejemplos típicos:

```python
df.select(...)
df.filter(...)
df.withColumn(...)
```

Conceptualmente:

```text
P0 ──► P0
P1 ──► P1
P2 ──► P2
```

Los datos pueden procesarse localmente dentro de cada partición.


### Wide Transformations

Una transformación es **wide** cuando los datos deben redistribuirse entre particiones.

Ejemplos frecuentes:

```python
df.groupBy(...)
df.join(...)
df.distinct()
df.repartition(...)
```

Conceptualmente:

```text
P0 ──┐
P1 ──┼──► Shuffle ──► P0'
P2 ──┼──────────────► P1'
P3 ──┘──────────────► P2'
```

Estas operaciones suelen ser más costosas porque pueden provocar un shuffle.

### 19.1 Comparación rápida


| Operación | Tipo habitual | Shuffle |
|---|---|---|
| `select()` | Narrow | No |
| `filter()` | Narrow | No |
| `withColumn()` | Narrow | No |
| `map()` | Narrow | No |
| `groupBy()` | Wide | Sí |
| `repartition()` | Wide | Sí |
| `distinct()` | Wide | Generalmente sí |
| `join()` | Depende del plan | Frecuentemente sí |

La distinción es importante porque:

```text
Narrow Transformation
        ↓
procesamiento local
        ↓
menor movimiento de datos


Wide Transformation
        ↓
shuffle
        ↓
mayor costo potencial
```


## 20. Ejecturar la agregación

Ahora sí solicitaremos el resultado:

In [45]:
df_por_estado.show(truncate=False)

+----------+-------------+------------------+
|estado    |num_registros|ingreso_promedio  |
+----------+-------------+------------------+
|CDMX      |3            |22666.666666666668|
|Jalisco   |2            |23250.0           |
|Nuevo León|2            |26000.0           |
|Puebla    |1            |27000.0           |
+----------+-------------+------------------+



Con esto Spark ejecutará el plan necesario para realizar la agregación.

Nuestro proceso conceptual es:

```text
DataFrame
    ↓
groupBy("estado")
    ↓
Shuffle
    ↓
Agregación
    ↓
show()
```

## 21. De una Action a un Job

Cuando una Action necesita obtener un resultado, Spark crea trabajo para ejecutar el plan.

De forma simplificada:

```text
Action
  ↓
Job
```

Por ejemplo:


In [46]:
df_por_estado.show()

+----------+-------------+------------------+
|    estado|num_registros|  ingreso_promedio|
+----------+-------------+------------------+
|      CDMX|            3|22666.666666666668|
|   Jalisco|            2|           23250.0|
|Nuevo León|            2|           26000.0|
|    Puebla|            1|           27000.0|
+----------+-------------+------------------+



puede provocar la creación de un Job.

Otro:

In [47]:
df_por_estado.count()

4

puede provocar otro Job.

La relación exacta puede variar dependiendo de la operación y de las optimizaciones que aplique Spark, pero esta representación nos servirá como modelo inicial.

## 22. Jobs, Stages y Tasks

Spark divide la ejecución en diferentes niveles.

Una forma útil de visualizarlo es:

```text
Spark Application
       │
       ├── Job
       │    │
       │    ├── Stage
       │    │    ├── Task
       │    │    ├── Task
       │    │    └── Task
       │    │
       │    └── Stage
       │         ├── Task
       │         └── Task
       │
       └── Job
```

Vamos a entender cada nivel.


### 22.1 Aplication

La **Application** corresponde a nuestra aplicación completa de Spark.

En nuestro notebook comenzó cuando creamos:

```python
spark = (
    SparkSession.builder
    .appName("02_01_spark_fundamentals")
    .getOrCreate()
)
```

Actualmente nuestra aplicación tiene un identificador parecido a:

```text
local-...
```

### 22.2 Job

Un **Job** representa el conjunto de trabajo que Spark necesita realizar como consecuencia de una Action.

Por ejemplo:

```python
df.count()
```

```python
df.show()
```

```python
df.collect()
```

pueden desencadenar Jobs.

### 22.3 Stage

Un Job puede dividirse en varios **Stages**.

Una causa muy importante de esta separación son los límites producidos por un **shuffle**.

Por ejemplo:

```text
Stage 1
   │
   ├── filter
   ├── select
   └── preparación de agregación
            │
            ▼
         SHUFFLE
            │
            ▼
Stage 2
   │
   └── agregación final
```

Podemos pensar inicialmente que:

> Un shuffle suele marcar una frontera importante entre Stages.


### 22.4 Task

Las **Tasks** son las unidades individuales de trabajo que Spark envía para procesar particiones.

Conceptualmente:

```text
Stage
 │
 ├── Partition 0 ──► Task 0
 ├── Partition 1 ──► Task 1
 ├── Partition 2 ──► Task 2
 └── Partition 3 ──► Task 3
```

Por eso existe una relación tan importante entre:

```text
Partitions
    ↕
Tasks
```

Dentro de un Stage, Spark normalmente crea una Task por partición que debe procesarse.

## 23. Relación completa

Ahora podemos conectar casi todos los conceptos estudiados.

```text
PySpark code
      │
      ▼
Transformations
      │
      ▼
Logical Plan
      │
      ▼
Optimized Logical Plan
      │
      ▼
Physical Plan
      │
      ▼
Action
      │
      ▼
Job
      │
      ▼
Stages
      │
      ▼
Tasks
      │
      ▼
Partitions
```

Si aparece una Wide Transformation:

```text
Stage
  │
  ▼
Shuffle
  │
  ▼
nuevo Stage
```

Este modelo mental será muy útil para comprender problemas de rendimiento posteriormente.



## 24. Experimento final: identificar un shuffle

Construiremos un pequeño pipeline que combine transformaciones narrow y wide.

In [48]:
df_experimento = (
    df
    .filter(F.col("edad") >= 30)
    .withColumn(
        "ingreso_anual",
        F.col("ingreso") * 12
    )
    .groupBy("estado")
    .agg(
        F.count("*").alias("personas"),
        F.avg("ingreso_anual").alias("ingreso_anual_promedio")
    )
)

Podemos clasificar las operaciones:

```text
filter()
    ↓
NARROW

withColumn()
    ↓
NARROW

groupBy()
    ↓
WIDE
    ↓
SHUFFLE
```

Ahora inspeccionaremos el plan:

In [49]:
df_experimento.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [4]: [id#0L, estado#1, edad#2L, ingreso#3]
Arguments: [id#0L, estado#1, edad#2L, ingreso#3], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [4]: [id#0L, estado#1, edad#2L, ingreso#3]
Condition : (isnotnull(edad#2L) AND (edad#2L >= 30))

(3) Project
Output [2]: [estado#1, (ingreso#3 * 12.0) AS ingreso_anual#202]
Input [4]: [id#0L, estado#1, edad#2L, ingreso#3]

(4) HashAggregate
Input [2]: [estado#1, ingreso_anual#202]
Keys [1]: [estado#1]
Functions [2]: [partial_count(1), partial_avg(ingreso_anual#202)]
Aggregate Attributes [3]: [count#212L, sum#213, count#214L]
Results [4]: [estado#1, count#215L, sum#216, count#217L]

(5) Exchange
Input [4]: [estado#1, count#215L, sum#216, 

Busca específicamente:

```text
Filter
Project
Exchange
HashAggregate
```

La presencia de:

```text
Exchange
```

nos ayudará a identificar el punto en el que Spark necesita redistribuir datos.

Finalmente:

In [50]:
df_experimento.show(truncate=False)

+----------+--------+----------------------+
|estado    |personas|ingreso_anual_promedio|
+----------+--------+----------------------+
|Jalisco   |2       |279000.0              |
|Nuevo León|2       |312000.0              |
|Puebla    |1       |324000.0              |
|CDMX      |1       |420000.0              |
+----------+--------+----------------------+



## 25. Conclusiones sobre el modelo de ejecución

Hasta este punto ya podemos construir una visión más completa de cómo trabaja Spark.

Una operación como:

```python
df.filter(...)
  .withColumn(...)
  .groupBy(...)
  .agg(...)
```

no debe interpretarse simplemente como una secuencia de instrucciones Python.

Spark:

1. registra las transformaciones;
2. construye un plan lógico;
3. optimiza el plan;
4. genera un plan físico;
5. espera una Action;
6. crea el trabajo necesario;
7. divide el trabajo en Stages;
8. ejecuta Tasks sobre las particiones.

Además, ahora podemos comenzar a reconocer una señal especialmente importante dentro de un plan:

```text
Exchange
```

que generalmente indica movimiento de datos entre particiones.

Esto será fundamental cuando comencemos a optimizar aplicaciones Spark.

## 26. Reutilización de resultados: `cache()` y `persist()`

Anteriormente observamos que dos Actions diferentes pueden provocar que Spark vuelva a calcular un mismo pipeline.

Por ejemplo:

```python
pipeline.show()

pipeline.count()
```

Si el DataFrame requiere transformaciones costosas y vamos a reutilizarlo varias veces, repetir todo el cálculo puede resultar innecesario.

Spark permite conservar el resultado de un DataFrame mediante:

- `cache()`
- `persist()`

Conceptualmente:

```text
SIN CACHE

Datos
  ↓
Transformaciones
  ↓
Action 1

Datos
  ↓
Transformaciones
  ↓
Action 2


CON CACHE

Datos
  ↓
Transformaciones
  ↓
Resultado almacenado
     ├──► Action 1
     ├──► Action 2
     └──► Action 3
```


## 27. Utilizar `cache()`

Vamos a utilizar el pipeline construido anteriormente.

In [51]:
pipeline_cache = pipeline.cache()

`cache()` indica a Spark que queremos conservar el resultado para reutilizarlo posteriormente.

Podemos consultar el nivel de almacenamiento:

In [52]:
print(pipeline_cache.storageLevel)

Disk Memory Deserialized 1x Replicated


En nuestra versión de Spark, `cache()` utiliza el nivel de almacenamiento predeterminado:

```text
MEMORY_AND_DISK_DESER
```

Esto significa, de forma simplificada, que Spark intentará conservar los datos utilizando memoria y podrá utilizar disco cuando sea necesario.

### 27.1 `cache()` también sigue el modelo Lazy

Es importante notar que:

```python
pipeline.cache()
```

**no significa que Spark haya calculado inmediatamente todo el DataFrame.**

Lo que hemos hecho es indicar:

> Cuando este DataFrame sea calculado, conserva su resultado para poder reutilizarlo.

Necesitamos una Action para materializarlo.

Por ejemplo:

In [53]:
pipeline_cache.count()

6

Esta primera Action:

```text
Transformaciones
      ↓
    count()
      ↓
ejecución del pipeline
      ↓
resultado almacenado
```

A partir de este momento, otras operaciones pueden reutilizar los datos almacenados.

Por ejemplo:

In [54]:
pipeline_cache.show(truncate=False)

+---+----------+----+-----------+-------+-------------+
|id |estado    |edad|grupo_edad |ingreso|ingreso_anual|
+---+----------+----+-----------+-------+-------------+
|2  |Jalisco   |34  |Menor de 35|22000.0|264000.0     |
|3  |Nuevo León|41  |35-44      |31000.0|372000.0     |
|5  |Puebla    |52  |45 o más   |27000.0|324000.0     |
|6  |Jalisco   |38  |35-44      |24500.0|294000.0     |
|7  |CDMX      |45  |45 o más   |35000.0|420000.0     |
|8  |Nuevo León|31  |Menor de 35|21000.0|252000.0     |
+---+----------+----+-----------+-------+-------------+



Conceptualmente:

```text
Primera Action

Datos originales
      ↓
Transformaciones
      ↓
Resultado
      ↓
     CACHE
      ↓
    count()


Segunda Action

CACHE
  ↓
show()
```

### 27.2 Observar el cache en el plan de ejecución

Podemos inspeccionar nuevamente el plan:

In [55]:
pipeline_cache.explain("formatted")

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [4]: [id#0L, estado#1, edad#2L, ingreso#3]
Arguments: [id#0L, estado#1, edad#2L, ingreso#3], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [4]: [id#0L, estado#1, edad#2L, ingreso#3]
Condition : (isnotnull(ingreso#3) AND (ingreso#3 > 20000.0))

(3) Project [codegen id : 1]
Output [6]: [id#0L, estado#1, edad#2L, CASE WHEN (edad#2L < 35) THEN Menor de 35 WHEN (edad#2L < 45) THEN 35-44 ELSE 45 o más END AS grupo_edad#48, ingreso#3, (ingreso#3 * 12.0) AS ingreso_anual#47]
Input [4]: [id#0L, estado#1, edad#2L, ingreso#3]




Entre las operaciones podemos encontrar referencias similares a:

```text
InMemoryRelation
```

o:

```text
InMemoryTableScan
```

Estas operaciones indican que Spark puede obtener los datos desde el resultado almacenado en lugar de reconstruir todo el pipeline desde su origen.

No es necesario memorizar los nombres exactos, pero debemos comenzar a asociar:

```text
InMemory...
     ↓
datos persistidos / cacheados
```

## 28. `persist()`

`persist()` cumple una función similar a `cache()`, pero nos permite seleccionar explícitamente cómo queremos almacenar los datos.

Primero liberaremos el cache anterior:

In [56]:
pipeline_cache.unpersist()

DataFrame[id: bigint, estado: string, edad: bigint, grupo_edad: string, ingreso: double, ingreso_anual: double]

Después importaremos `StorageLevel`:

In [57]:
from pyspark import StorageLevel

Podemos persistir el DataFrame únicamente en disco:

In [58]:
pipeline_persist = pipeline.persist(
    StorageLevel.DISK_ONLY
)

Consultamos el nivel utilizado:

In [59]:
print(pipeline_persist.storageLevel)

Disk Serialized 1x Replicated


Y materializamos el DataFrame con una Action:

In [60]:
pipeline_persist.count()

6

### 28.1 Algunos niveles de persistencia

Spark ofrece diferentes estrategias de almacenamiento.

Algunos ejemplos importantes son:

```python
StorageLevel.MEMORY_ONLY

StorageLevel.MEMORY_AND_DISK

StorageLevel.MEMORY_AND_DISK_DESER

StorageLevel.DISK_ONLY
```

Conceptualmente:

```text
MEMORY_ONLY
    ↓
prioriza memoria


MEMORY_AND_DISK
    ↓
memoria + respaldo en disco


DISK_ONLY
    ↓
almacenamiento en disco
```

La elección depende de factores como:

- tamaño del DataFrame;
- memoria disponible;
- costo de volver a calcularlo;
- frecuencia con la que será reutilizado.

Por ahora no necesitamos optimizar estos niveles manualmente. Lo importante es entender que Spark nos permite decidir cómo conservar resultados intermedios.

## 29. `cache()` vs `persist()`

La principal diferencia práctica es el nivel de control.

| Característica | `cache()` | `persist()` |
|---|---|---|
| Conserva resultados | Sí | Sí |
| Permite elegir StorageLevel | No directamente | Sí |
| Uso sencillo | Sí | Sí |
| Configuración personalizada | No | Sí |

Podemos pensar inicialmente en:

```text
Quiero reutilizar este DataFrame
y no necesito controlar el almacenamiento
             ↓
          cache()


Quiero controlar cómo se almacena
             ↓
         persist()
```

## 30. Liberar los datos almacenados

Mantener datos persistidos también consume recursos.

Cuando ya no necesitamos reutilizar un DataFrame debemos poder liberar ese almacenamiento.

Para ello utilizamos:

In [61]:
pipeline_persist.unpersist()

DataFrame[id: bigint, estado: string, edad: bigint, grupo_edad: string, ingreso: double, ingreso_anual: double]

Podemos comprobar nuevamente su nivel:

In [62]:
print(pipeline_persist.storageLevel)

Serialized 1x Replicated


Esto es especialmente importante en aplicaciones grandes, donde mantener numerosos DataFrames almacenados puede ejercer presión sobre la memoria y el disco disponibles.

Conceptualmente:

```text
cache / persist
      ↓
consumo de recursos
      ↓
reutilización
      ↓
trabajo terminado
      ↓
unpersist()
```

## 31. ¿Cuándo conviene utilizar cache?

No debemos utilizar `cache()` automáticamente sobre todos los DataFrames.

Tiene sentido principalmente cuando un resultado:

1. es costoso de calcular;
2. será utilizado varias veces;
3. cabe razonablemente en los recursos disponibles;
4. evita repetir transformaciones significativas.

Por ejemplo:

```text
Dataset grande
    ↓
limpieza
    ↓
joins
    ↓
transformaciones costosas
    ↓
DataFrame reutilizado
    ├──► análisis 1
    ├──► análisis 2
    └──► modelo
```

En este caso podría ser conveniente almacenar el resultado intermedio.

En cambio:

```text
Dataset
   ↓
transformación sencilla
   ↓
una única Action
```

probablemente no necesite cache.

La idea importante es:

> Cachear también tiene un costo. Debemos hacerlo cuando evitar el recálculo compense el almacenamiento adicional.

## 32. Modelo mental completo de Spark

Con los conceptos estudiados en este notebook podemos construir nuestro primer modelo general de funcionamiento de Spark:

```text
                    PYSPARK
                       │
                       ▼
                SparkSession
                       │
                       ▼
                   DataFrame
                       │
                       ▼
               Transformations
                       │
                       ▼
               Lazy Evaluation
                       │
                       ▼
                 Logical Plan
                       │
                       ▼
            Optimized Logical Plan
                       │
                       ▼
                 Physical Plan
                       │
                       ▼
                    Action
                       │
                       ▼
                     Job
                       │
                       ▼
                    Stages
                       │
             ┌─────────┴─────────┐
             │                   │
       Narrow operations      Shuffle
             │                   │
             │                Exchange
             │                   │
             └─────────┬─────────┘
                       ▼
                     Tasks
                       │
                       ▼
                  Partitions
                       │
                       ▼
                   Resultado
                       │
                       ▼
              cache / persist
              si se reutiliza
```

## 33. Conclusiones

En este notebook estudiamos los fundamentos necesarios para comenzar a comprender Spark como un motor de procesamiento distribuido.

Aprendimos que un DataFrame de Spark no debe entenderse únicamente como una tabla, sino como una estructura cuyo procesamiento puede distribuirse entre múltiples particiones.

También vimos que Spark utiliza **Lazy Evaluation**, lo que le permite analizar y optimizar un conjunto de transformaciones antes de comenzar la ejecución.

Relacionamos posteriormente:

```text
Transformations
      ↓
Action
      ↓
Job
      ↓
Stages
      ↓
Tasks
      ↓
Partitions
```

y observamos que ciertas operaciones pueden provocar un **shuffle**, visible en los planes físicos mediante operaciones como `Exchange`.

Finalmente, introdujimos `cache()` y `persist()` como mecanismos para evitar cálculos repetidos cuando un mismo resultado será reutilizado.

Los conceptos principales que debemos conservar de este notebook son:

| Concepto | Idea principal |
|---|---|
| `SparkSession` | Punto de entrada para trabajar con Spark |
| DataFrame | Estructura distribuida de datos |
| Schema | Define estructura y tipos |
| Transformation | Describe una transformación |
| Action | Solicita un resultado |
| Lazy Evaluation | Retrasa la ejecución |
| Partition | Unidad lógica de distribución de datos |
| Task | Trabajo realizado sobre una partición |
| Stage | Conjunto de Tasks sin una frontera de shuffle |
| Job | Trabajo generado por una Action |
| Shuffle | Redistribución de datos entre particiones |
| `Exchange` | Señal frecuente de shuffle en un plan |
| `repartition()` | Redistribuye los datos entre particiones |
| `coalesce()` | Reduce particiones evitando normalmente una redistribución completa |
| `cache()` | Conserva un DataFrame para reutilizarlo |
| `persist()` | Permite controlar el nivel de almacenamiento |
| `unpersist()` | Libera los datos persistidos |

Con esta base podemos dejar de concentrarnos únicamente en **cómo funciona Spark** y comenzar a utilizar de manera más completa la API de DataFrames.